In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn import datasets
from sklearn.cluster import KMeans
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage

## Zadanie 1
    
Wczytaj dane z pliku **PersonalData.csv**.
    
Oblicz odległość euklidesową \(d(A, B)\) między dwoma wybranymi rekordami \(A\) i \(B\).
    
Zmodyfikuj skalę dla atrybutu **zarobki** (przeskaluj jednostki z tysięcy na złote) i ponownie oblicz odległość euklidesową. Jak zmiana skali wpływa na wynik?
    
Aby wyeliminować wpływ skali, podziel wartość każdego atrybutu przez jego odchylenie standardowe. Oblicz ponownie odległość euklidesową dla przeskalowanych danych.
    
Dla punktów \(A\) i \(B\) oblicz także odległość Minkowskiego oraz odległość miejską (Manhattan).

In [ ]:
df = pd.read_csv("PersonalData.csv", index_col=0, delimiter=",")
num = df.select_dtypes(include="number")
A = num.iloc[0].to_numpy()
B = num.iloc[1].to_numpy()
d_euc = float(np.linalg.norm(A - B))
d_euc


In [ ]:
num2 = num.copy()
if "zarobki" in num2.columns:
    num2["zarobki"] = num2["zarobki"] * 1000
A2 = num2.iloc[0].to_numpy()
B2 = num2.iloc[1].to_numpy()
d_euc_scaled = float(np.linalg.norm(A2 - B2))
d_euc_scaled


In [ ]:
std = num.std(ddof=0).replace(0, np.nan)
num_std = num.div(std, axis=1).dropna(axis=1, how="any")
A3 = num_std.iloc[0].to_numpy()
B3 = num_std.iloc[1].to_numpy()
d_euc_std = float(np.linalg.norm(A3 - B3))
d_euc_std


In [ ]:
p = 3
d_minkowski = float(np.power(np.sum(np.abs(A - B) ** p), 1/p))
d_manhattan = float(np.sum(np.abs(A - B)))
{"minkowski_p3": d_minkowski, "manhattan": d_manhattan}


## Zadanie 2

Masz opisy trzech filmów
- `film1 = "kosmiczna stacja astronauta planeta obca"`
- `film2 = "astronauta rakieta księżyc misja kosmiczna"`
- `film3 = "wampir zamek noc krew mroczny"`

a. Stwórz reprezentację wektorową dla każdego filmu używając CountVectorizer

b. Oblicz podobieństwo cosinusowe między wszystkimi parami filmów

c. Które dwa filmy są najbardziej podobne? Które najmniej?

d . Dodaj czwarty film i sprawdź do którego jest najbardziej podobny:
    `film4 = "rakieta mars astronauta czerwona planeta"`

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

film1 = "kosmiczna stacja astronauta planeta obca"
film2 = "astronauta rakieta księżyc misja kosmiczna"
film3 = "wampir zamek noc krew mroczny"

corpus = [film1, film2, film3]
vec = CountVectorizer()
X = vec.fit_transform(corpus)
pd.DataFrame(X.toarray(), columns=vec.get_feature_names_out(), index=["film1","film2","film3"])


In [ ]:
sim = cosine_similarity(X)
sim_df = pd.DataFrame(sim, index=["film1","film2","film3"], columns=["film1","film2","film3"])
sim_df


In [ ]:
pairs = []
names = ["film1","film2","film3"]
for i in range(3):
    for j in range(i+1,3):
        pairs.append(((names[i], names[j]), float(sim[i,j])))
pairs_sorted = sorted(pairs, key=lambda x: x[1], reverse=True)
pairs_sorted, pairs_sorted[0], pairs_sorted[-1]


In [ ]:
film4 = "rakieta mars astronauta czerwona planeta"
corpus2 = [film1, film2, film3, film4]
X2 = vec.fit_transform(corpus2)
sim2 = cosine_similarity(X2)
sim2_df = pd.DataFrame(sim2, index=["film1","film2","film3","film4"], columns=["film1","film2","film3","film4"])
sim2_df.loc["film4"].drop("film4").sort_values(ascending=False)


## Zadanie 3

1. Załaduj zbiór danych **iris**.
Opis zbioru: https://scikit-learn.org/stable/auto_examples/datasets/plot_iris_dataset.html
2. Narysuj wykres rozrzutu dla współrzędnych **sepal_length** i **sepal_width**. Oznacz różnymi
kolorami i marklami różne klasy kwiatów.
3. Znajdź wartości **minimalne, maksymalne** oraz **średnie odchylenie kwadratowe** dla atrybutów zbioru.
4. Policz **współczynniki korelacji** między atrybutami.
5. Zastosuj **algorytm k-średnich** i znajdź podział na klastry dla współrzędnych **sepal_length** i **sepal_width**.
6. Zastosuj **algorytm hierarchiczny aglomeracyjny** i znajdź podział na klastry dla współrzędnych **sepal_length** i **sepal_width**.

In [ ]:
iris = datasets.load_iris()
df_iris = pd.DataFrame(iris.data, columns=iris.feature_names)
df_iris["target"] = iris.target
df_iris["class"] = [iris.target_names[i] for i in iris.target]
df_iris.head()


In [ ]:
import matplotlib.pyplot as plt

plt.figure()
for cls, marker in zip(df_iris["class"].unique(), ["o","s","^"]):
    sub = df_iris[df_iris["class"] == cls]
    plt.scatter(sub["sepal length (cm)"], sub["sepal width (cm)"], marker=marker, label=cls)
plt.xlabel("sepal length (cm)")
plt.ylabel("sepal width (cm)")
plt.title("Iris: sepal length vs sepal width")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
stats_tbl = pd.DataFrame({
    "min": df_iris[iris.feature_names].min(),
    "max": df_iris[iris.feature_names].max(),
    "std": df_iris[iris.feature_names].std(ddof=0)
})
stats_tbl


In [ ]:
corr = df_iris[iris.feature_names].corr()
corr


In [ ]:
X_sw = df_iris[["sepal length (cm)", "sepal width (cm)"]].to_numpy()
km = KMeans(n_clusters=3, random_state=42, n_init=10)
labels_km = km.fit_predict(X_sw)

plt.figure()
plt.scatter(X_sw[:,0], X_sw[:,1], c=labels_km)
plt.xlabel("sepal length (cm)")
plt.ylabel("sepal width (cm)")
plt.title("KMeans (k=3) na sepal length/width")
plt.tight_layout()
plt.show()


In [ ]:
agg = AgglomerativeClustering(n_clusters=3, linkage="ward")
labels_ag = agg.fit_predict(X_sw)

plt.figure()
plt.scatter(X_sw[:,0], X_sw[:,1], c=labels_ag)
plt.xlabel("sepal length (cm)")
plt.ylabel("sepal width (cm)")
plt.title("Aglomeracyjne (ward, k=3) na sepal length/width")
plt.tight_layout()
plt.show()


In [ ]:
# Wczytanie zbioru danych Iris
iris = datasets.load_iris()

## Zadanie 4
Dla danych z pliku penguins.csv wykonaj polecenia

Wyświetl podstawowe informacje o zbiorze.
1. Sprawdź czy w zbiorze nie brakuje danych. Jeżeli są – usuń je ().
2. Ogranicz się do atrybutów **bill_length_mm** i **flipper_length_mm**.
3. Narysuj dendogram, zinterpretuj go i wyznacz ilość klastrów.
4. Zastosuj algorytm hierarchiczny aglomeracyjny do zbioru i wyznacz klastry.
5. Stwórz wykres rozrzutu z zaznaczonymi klastrami.

W **punktach 3-5** przetestuj różne sposoby obliczenia odległości między klastrami (linkage-complete, average, single, ward).

Dane z https://github.com/mwaskom/seaborn-data/blob/master/penguins.csv

In [ ]:
peng = pd.read_csv("penguins.csv")
peng = peng.dropna()
peng2 = peng[["bill_length_mm", "flipper_length_mm"]].copy()
peng2.head()


In [ ]:
import matplotlib.pyplot as plt

X = peng2.to_numpy()
methods = ["single", "complete", "average", "ward"]
for m in methods:
    Z = linkage(X, method=m, metric="euclidean")
    plt.figure(figsize=(10,4))
    dendrogram(Z, truncate_mode="lastp", p=30)
    plt.title(f"Dendrogram (linkage={m})")
    plt.xlabel("klastry")
    plt.ylabel("odległość")
    plt.tight_layout()
    plt.show()


In [ ]:
from sklearn.cluster import AgglomerativeClustering
import matplotlib.pyplot as plt

X = peng2.to_numpy()
methods = ["single", "complete", "average", "ward"]
k = 3

for m in methods:
    model = AgglomerativeClustering(n_clusters=k, linkage=m)
    lab = model.fit_predict(X)
    plt.figure()
    plt.scatter(X[:,0], X[:,1], c=lab)
    plt.xlabel("bill_length_mm")
    plt.ylabel("flipper_length_mm")
    plt.title(f"Aglomeracyjne: linkage={m}, k={k}")
    plt.tight_layout()
    plt.show()


In [ ]:
df = pd.read_csv("PersonalData.csv", index_col=0, delimiter=",")

In [ ]:
df

,Wzrost (cm),Waga (kg),Staz (lata),Zarobki (tys.),Ocena (pkt.),Piętro,Dzieci,Odleglosc (km),Ubezp.
A,190,88,3,3.5,7,6,1,25,Tak
B,172,70,12,4.3,5,1,4,12,Nie


In [ ]:
df.loc['A']

,A
Wzrost (cm),190
Waga (kg),88
Staz (lata),3
Zarobki (tys.),3.5
Ocena (pkt.),7
Piętro,6
Dzieci,1
Odleglosc (km),25
Ubezp.,Tak
